# DataCo Smart Supply Chain — exploration + classical baseline

External real-world dataset (separate from the synthetic `scm_dataset` benchmark) used as a second track: reproduce a legitimate classical baseline here, then later build a QGNN on the same graph/split for a direct comparison.

**Source**: Constante, Silva & Pereira (2019), Mendeley Data V5, DOI `10.17632/8gx2fvg2k6.5`, CC-BY-4.0. Downloaded via `scripts/download_dataco.py` -> `data/raw/dataco/`. Provenance: `data/raw/dataco/provenance.json`.

**Why not reproduce the "Integrated Machine Learning for Enhanced Supply Chain Risk Prediction" preprint (Jin, 2025, Preprints.org, not peer-reviewed)**: its Figure 4 lists features ("Lead Time", "Demand Variation", "Supplier Reliability", "Inventory Level", "Order Cost") that **do not exist** in the actual public DataCo schema (verified below) despite the paper claiming to use DataCo Global's data. That mismatch, combined with the "not peer-reviewed" status, means its reported 85.4% accuracy / 0.85 F1 isn't a trustworthy reproduction target. This notebook builds an honest baseline from the real columns instead, and treats peer-reviewed / independently-reproducible numbers (e.g. the EAGLE paper's F1=0.876 graph-based result, or a plain from-scratch Random Forest) as the actual comparison points.

In [2]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/raw/dataco/DataCoSupplyChainDataset.csv"
df = pd.read_csv(DATA_PATH, encoding="latin-1")
df.shape

(180519, 53)

## 1. Schema audit

Confirm the real columns (this is what caught the preprint's fabricated feature list above) and check the target's class balance.

In [3]:
list(df.columns)

['Type',
 'Days for shipping (real)',
 'Days for shipment (scheduled)',
 'Benefit per order',
 'Sales per customer',
 'Delivery Status',
 'Late_delivery_risk',
 'Category Id',
 'Category Name',
 'Customer City',
 'Customer Country',
 'Customer Email',
 'Customer Fname',
 'Customer Id',
 'Customer Lname',
 'Customer Password',
 'Customer Segment',
 'Customer State',
 'Customer Street',
 'Customer Zipcode',
 'Department Id',
 'Department Name',
 'Latitude',
 'Longitude',
 'Market',
 'Order City',
 'Order Country',
 'Order Customer Id',
 'order date (DateOrders)',
 'Order Id',
 'Order Item Cardprod Id',
 'Order Item Discount',
 'Order Item Discount Rate',
 'Order Item Id',
 'Order Item Product Price',
 'Order Item Profit Ratio',
 'Order Item Quantity',
 'Sales',
 'Order Item Total',
 'Order Profit Per Order',
 'Order Region',
 'Order State',
 'Order Status',
 'Order Zipcode',
 'Product Card Id',
 'Product Category Id',
 'Product Description',
 'Product Image',
 'Product Name',
 'Product P

In [4]:
df["Late_delivery_risk"].value_counts(normalize=True)

Late_delivery_risk
1    0.548291
0    0.451709
Name: proportion, dtype: float64

## 2. Feature audit (leakage prevention)

Same principle as `feature_audit.csv` in the synthetic benchmark (plan §27): mark every column allowed/disallowed as a model input, with a reason, *before* touching modeling code.

**Disallowed (label / post-outcome leakage)**
- `Late_delivery_risk` -- the target itself
- `Delivery Status` -- directly encodes whether delivery was late (near-tautological with the target)
- `Days for shipping (real)` -- the *actual* transit time, only known after delivery completes
- `shipping date (DateOrders)` -- the actual ship date, also only known after the fact
- `Order Status` -- fulfillment status observed progressively after order placement (e.g. `CANCELED`, `SUSPECTED_FRAUD`), not known at prediction time

**Disallowed (PII / identifiers, never modeled on regardless of predictive value)**
- `Customer Email`, `Customer Fname`, `Customer Lname`, `Customer Password`, `Customer Street`
- `Product Description`, `Product Image` (free text / URLs, no modeling value)

**Allowed (known at order time)**
- `Type`, `Days for shipment (scheduled)`, `Category Id`, `Customer Segment`, `Customer Country`, `Customer State`, `Department Id`, `Market`, `Order Region`, `Order Country`, `Order State`, `Order Item Discount(...)`, `Order Item Product Price`, `Order Item Quantity`, `Sales`, `Product Price`, `Shipping Mode`, `order date (DateOrders)` (used for the temporal split, not as a feature itself)

In [5]:
TARGET = "Late_delivery_risk"

DISALLOWED = [
    "Delivery Status", "Days for shipping (real)", "shipping date (DateOrders)", "Order Status",
    "Customer Email", "Customer Fname", "Customer Lname", "Customer Password", "Customer Street",
    "Product Description", "Product Image",
]

CATEGORICAL_FEATURES = [
    "Type", "Category Id", "Customer Segment", "Customer Country", "Customer State",
    "Department Id", "Market", "Order Region", "Order Country", "Order State", "Shipping Mode",
]
NUMERIC_FEATURES = [
    "Days for shipment (scheduled)", "Order Item Discount", "Order Item Discount Rate",
    "Order Item Product Price", "Order Item Quantity", "Sales", "Product Price",
]

assert TARGET not in CATEGORICAL_FEATURES + NUMERIC_FEATURES
assert not set(DISALLOWED) & set(CATEGORICAL_FEATURES + NUMERIC_FEATURES)
df[["order date (DateOrders)", TARGET] + CATEGORICAL_FEATURES + NUMERIC_FEATURES].isna().sum()

order date (DateOrders)          0
Late_delivery_risk               0
Type                             0
Category Id                      0
Customer Segment                 0
Customer Country                 0
Customer State                   0
Department Id                    0
Market                           0
Order Region                     0
Order Country                    0
Order State                      0
Shipping Mode                    0
Days for shipment (scheduled)    0
Order Item Discount              0
Order Item Discount Rate         0
Order Item Product Price         0
Order Item Quantity              0
Sales                            0
Product Price                    0
dtype: int64

## 3. Temporal split

Chronological, not random-shuffled (same reasoning as the synthetic benchmark's temporal split -- a random row shuffle of ordered transactions leaks future information into training).

In [6]:
df["order date (DateOrders)"] = pd.to_datetime(df["order date (DateOrders)"])
df_sorted = df.sort_values("order date (DateOrders)").reset_index(drop=True)

n = len(df_sorted)
train_end, val_end = int(n * 0.70), int(n * 0.85)
df_sorted["split"] = "test"
df_sorted.loc[:train_end, "split"] = "train"
df_sorted.loc[train_end:val_end, "split"] = "validation"
df_sorted["split"].value_counts()

split
train         126363
validation     27079
test           27077
Name: count, dtype: int64

### 3a. Split balance diagnostic

Confirms the temporal split above isn't hiding a distribution shift -- if the positive rate or date range looks wildly different across splits, the F1/accuracy numbers in section 4 aren't directly comparable across train/val/test, and any QGNN result built on this same split inherits the same distortion (see discussion: this split is reused unchanged for the QGNN graph construction in section 5).

In [7]:
split_summary = df_sorted.groupby("split").agg(
    n=(TARGET, "size"),
    positive_rate=(TARGET, "mean"),
    date_min=("order date (DateOrders)", "min"),
    date_max=("order date (DateOrders)", "max"),
).loc[["train", "validation", "test"]]
split_summary

,n,positive_rate,date_min,date_max
split,,,,
train,126363,0.549425,2015-01-01 00:00:00,2017-01-08 02:49:00
validation,27079,0.541083,2017-01-08 02:49:00,2017-06-14 07:19:00
test,27077,0.550209,2017-06-14 07:19:00,2018-01-31 23:38:00


## 4. Preprocessing + classical baselines

Logistic Regression (matches the synthetic benchmark's non-graph baseline in `src/scm_dataset/modeling/baselines.py`) and Random Forest (the strongest single classical model reported across the DataCo papers surveyed -- e.g. the GitHub RF reproduction, and a component of the ensemble papers). `class_weight="balanced"` mirrors the existing project's imbalance handling, though DataCo's ~55/45 split is much less imbalanced than the synthetic supplier-disruption task.

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ("num", StandardScaler(), NUMERIC_FEATURES),
])

train = df_sorted[df_sorted["split"] == "train"]
val = df_sorted[df_sorted["split"] == "validation"]
test = df_sorted[df_sorted["split"] == "test"]

X_train = preprocessor.fit_transform(train[CATEGORICAL_FEATURES + NUMERIC_FEATURES])
X_val = preprocessor.transform(val[CATEGORICAL_FEATURES + NUMERIC_FEATURES])
X_test = preprocessor.transform(test[CATEGORICAL_FEATURES + NUMERIC_FEATURES])
y_train, y_val, y_test = train[TARGET].values, val[TARGET].values, test[TARGET].values

In [19]:
def evaluate(name, model, X, y):
    probs = model.predict_proba(X)[:, 1]
    preds = (probs >= 0.5).astype(int)
    return {
        "model": name,
        "accuracy": accuracy_score(y, preds),
        "precision": precision_score(y, preds),
        "recall": recall_score(y, preds),
        "f1": f1_score(y, preds),
        "roc_auc": roc_auc_score(y, probs),
    }

lr = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1).fit(X_train, y_train)

results = pd.DataFrame([
    evaluate("logistic_regression", lr, X_test, y_test),
    evaluate("random_forest", rf, X_test, y_test),
])
results

,model,accuracy,precision,recall,f1,roc_auc
0,logistic_regression,0.688407,0.824902,0.550544,0.660360,0.743260
1,random_forest,0.672933,0.750664,0.607263,0.671391,0.732852


## 5. Graph construction

DataCo has no native supplier network, so this graph is *derived*, not given -- the schema below is deliberately built to parallel the synthetic benchmark's node/edge design (`src/scm_dataset/schema/nodes.py`, `edges.py`): a bridging transactional node (`ProcurementOrder` there, `Order` here) connects the other entities, and edges follow the exact same `(src_type, edge_type, dst_type)` + `rev_<edge_type>` convention as `hetero_graph.py`, so a future GraphSAGE model over this graph can reuse that convolution code with minimal changes.

**Node types**
- `order` (180,519 -- one per order-item row, same unit as the classical baseline above, for an apples-to-apples comparison): the node actually classified. Features = the exact `CATEGORICAL_FEATURES`/`NUMERIC_FEATURES` from section 2 that are order-level rather than entity-level (`Type`, `Market`, `Shipping Mode`, `Days for shipment (scheduled)`, discount/price/quantity/sales). Label = `Late_delivery_risk`. `train`/`validation`/`test` masks reuse the identical chronological split from section 3 -- not recomputed.
- `customer` (20,652, keyed by `Customer Id`): `Customer Segment` + `Customer Country`, verified constant per customer (checked separately: 0/20,652 customers have more than one segment or country value).
- `product` (118, keyed by `Product Card Id`): `Category Id` + `Product Price`, verified constant per product.
- `region` (23, keyed by `Order Region` -- the shipping destination): **not** a static attribute -- its one feature is `historical_late_rate`, the mean `Late_delivery_risk` computed from **TRAIN orders only**, frozen and reused unchanged for validation/test (identical leakage discipline to `reduction.py`'s `PCASupplierReducer`/`DomainSelectedReducer`, and to the synthetic benchmark's own calibrated region-risk table). `Order Country`/`Order State` are left out as separate node types for now (164 and 1,089 distinct values -- too sparse to give a GNN useful structure at this stage).

**Edge types** (each with its reverse for bidirectional message passing, per `hetero_graph.py`'s convention)
- `(order, placed_by, customer)`
- `(order, contains, product)`
- `(order, ships_to, region)`

All encoders/scalers below are fit on **train order rows only**, exactly mirroring section 4's `preprocessor.fit_transform(train[...])` -- this is the same rule applied consistently, not a new one.

In [9]:
from sklearn.preprocessing import OneHotEncoder as OHE, StandardScaler as SS
import torch
from torch_geometric.data import HeteroData

ORDER_CATEGORICAL = ["Type", "Market", "Shipping Mode"]
ORDER_NUMERIC = [
    "Days for shipment (scheduled)", "Order Item Discount", "Order Item Discount Rate",
    "Order Item Product Price", "Order Item Quantity", "Sales",
]

train_mask_bool = (df_sorted["split"] == "train").values
val_mask_bool = (df_sorted["split"] == "validation").values
test_mask_bool = (df_sorted["split"] == "test").values

order_cat_encoder = OHE(handle_unknown="ignore").fit(df_sorted.loc[train_mask_bool, ORDER_CATEGORICAL])
order_num_scaler = SS().fit(df_sorted.loc[train_mask_bool, ORDER_NUMERIC])

order_x = np.hstack([
    order_cat_encoder.transform(df_sorted[ORDER_CATEGORICAL]).toarray(),
    order_num_scaler.transform(df_sorted[ORDER_NUMERIC]),
]).astype(np.float32)
order_y = df_sorted[TARGET].values.astype(np.float32)
print("order_x", order_x.shape, "order_y", order_y.shape)

order_x (180519, 19) order_y (180519,)


In [10]:
customer_attrs = df_sorted.drop_duplicates("Customer Id").set_index("Customer Id")
customer_ids = sorted(customer_attrs.index)
customer_id_to_idx = {cid: i for i, cid in enumerate(customer_ids)}
customer_attrs = customer_attrs.loc[customer_ids]

customer_cat_encoder = OHE(handle_unknown="ignore").fit(customer_attrs[["Customer Segment", "Customer Country"]])
customer_x = customer_cat_encoder.transform(customer_attrs[["Customer Segment", "Customer Country"]]).toarray().astype(np.float32)

product_attrs = df_sorted.drop_duplicates("Product Card Id").set_index("Product Card Id")
product_ids = sorted(product_attrs.index)
product_id_to_idx = {pid: i for i, pid in enumerate(product_ids)}
product_attrs = product_attrs.loc[product_ids]

product_cat_encoder = OHE(handle_unknown="ignore").fit(product_attrs[["Category Id"]])
product_price_scaler = SS().fit(product_attrs[["Product Price"]])
product_x = np.hstack([
    product_cat_encoder.transform(product_attrs[["Category Id"]]).toarray(),
    product_price_scaler.transform(product_attrs[["Product Price"]]),
]).astype(np.float32)

print("customer_x", customer_x.shape, "product_x", product_x.shape)

customer_x (20652, 5) product_x (118, 52)


In [11]:
region_ids = sorted(df_sorted["Order Region"].unique())
region_id_to_idx = {rid: i for i, rid in enumerate(region_ids)}

train_rows = df_sorted.loc[train_mask_bool]
region_train_stats = train_rows.groupby("Order Region")[TARGET].agg(["mean", "count"])
global_train_rate = train_rows[TARGET].mean()  # fallback for a region absent from train, should not occur here (23 regions, 126k train rows) but handled defensively

region_x = np.zeros((len(region_ids), 2), dtype=np.float32)
for rid, idx in region_id_to_idx.items():
    if rid in region_train_stats.index:
        rate = region_train_stats.loc[rid, "mean"]
        count = region_train_stats.loc[rid, "count"]
    else:
        rate, count = global_train_rate, 0
    region_x[idx] = [rate, np.log1p(count)]

assert (region_train_stats["count"] > 0).all(), "a region has zero train orders -- fallback path is untested territory, investigate before trusting it"
print("region_x", region_x.shape)
pd.DataFrame(region_x, index=region_ids, columns=["historical_late_rate", "log1p_train_count"])

region_x (23, 2)


,historical_late_rate,log1p_train_count
Canada,0.495495,6.790097
Caribbean,0.542131,8.346168
Central Africa,0.582327,7.361375
Central America,0.547611,9.543808
Central Asia,0.544355,6.208590
East Africa,0.561516,7.447751
East of USA,0.556616,8.841593
Eastern Asia,0.541485,8.691986
Eastern Europe,0.553022,8.200014
North Africa,0.547806,8.024535


In [12]:
n_orders = len(df_sorted)
customer_idx_per_order = df_sorted["Customer Id"].map(customer_id_to_idx).values
product_idx_per_order = df_sorted["Product Card Id"].map(product_id_to_idx).values
region_idx_per_order = df_sorted["Order Region"].map(region_id_to_idx).values
assert not np.any(pd.isna(customer_idx_per_order)) and not np.any(pd.isna(product_idx_per_order)) and not np.any(pd.isna(region_idx_per_order))

order_idx = np.arange(n_orders)

data = HeteroData()
data["order"].x = torch.tensor(order_x)
data["order"].y = torch.tensor(order_y)
data["order"].train_mask = torch.tensor(train_mask_bool)
data["order"].val_mask = torch.tensor(val_mask_bool)
data["order"].test_mask = torch.tensor(test_mask_bool)
data["customer"].x = torch.tensor(customer_x)
data["product"].x = torch.tensor(product_x)
data["region"].x = torch.tensor(region_x)

def add_relation(src_type, rel_name, dst_type, dst_idx_per_order):
    forward = torch.tensor(np.stack([order_idx, dst_idx_per_order]), dtype=torch.long)
    data[(src_type, rel_name, dst_type)].edge_index = forward
    data[(dst_type, f"rev_{rel_name}", src_type)].edge_index = forward.flip(0)

add_relation("order", "placed_by", "customer", customer_idx_per_order)
add_relation("order", "contains", "product", product_idx_per_order)
add_relation("order", "ships_to", "region", region_idx_per_order)

data.validate(raise_on_error=True)
print(data)

HeteroData(
  order={
    x=[180519, 19],
    y=[180519],
    train_mask=[180519],
    val_mask=[180519],
    test_mask=[180519],
  },
  customer={ x=[20652, 5] },
  product={ x=[118, 52] },
  region={ x=[23, 2] },
  (order, placed_by, customer)={ edge_index=[2, 180519] },
  (customer, rev_placed_by, order)={ edge_index=[2, 180519] },
  (order, contains, product)={ edge_index=[2, 180519] },
  (product, rev_contains, order)={ edge_index=[2, 180519] },
  (order, ships_to, region)={ edge_index=[2, 180519] },
  (region, rev_ships_to, order)={ edge_index=[2, 180519] }
)


In [13]:
assert data["order"].train_mask.sum().item() == (df_sorted["split"] == "train").sum()
assert data["order"].val_mask.sum().item() == (df_sorted["split"] == "validation").sum()
assert data["order"].test_mask.sum().item() == (df_sorted["split"] == "test").sum()
assert (data["order"].train_mask & data["order"].val_mask & data["order"].test_mask).sum().item() == 0
assert not torch.isnan(data["order"].x).any()
assert not torch.isnan(data["customer"].x).any()
assert not torch.isnan(data["product"].x).any()
assert not torch.isnan(data["region"].x).any()

import os
os.makedirs("../data/processed", exist_ok=True)
torch.save(data, "../data/processed/dataco_graph.pt")
print("saved to data/processed/dataco_graph.pt")

saved to data/processed/dataco_graph.pt


## 6. Next steps (not implemented yet)

- Train a heterogeneous GraphSAGE on this graph (`order` node classification, masks already set) as the graph-based classical reference point -- reusing `hetero_graph.py`'s relation-naming convention should make adapting the existing `HeteroGraphSAGE` architecture mostly a matter of swapping node/edge type names, not rewriting the model.
- Only after that: reduce the trained GraphSAGE's `order` embeddings to a small width (PCA or domain-selected, matching `reduction.py`'s pattern) and feed that into a `QuantumCircuitLayer` (`qgnn.py`) -- mirrors the existing QGNN-v2 architecture (frozen GraphSAGE embedding -> PCA -> quantum circuit) rather than the graph-agnostic QGNN-v1 ablation, since the whole motivation for building a real DataCo graph (vs. just quantum-encoding the section 4 tabular features) is to test whether the graph's structural signal survives being compressed into a QGNN.
- Compare final QGNN metrics against: this graph's own GraphSAGE result, the section 4 classical baselines (RF F1=0.671, ROC-AUC=0.733), and EAGLE's literature result (F1=0.876) as the external ceiling.